In [1]:

'''

GUILHERME LIMA DE SOUSA - 
ESTUDO DE CASO: TONELADAS TRANSPORTADAS EM FERROVIAS BRASILEIRAS
OBJETIVO: PREDIÇÃO USANDO RANDOMFORESTREGRESSOR

ENTRE EM CONTATO COMIGO NO LINKEDIN: 
www.linkedin.com/in/guilherme-lima-747355169

'''


import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns
from numpy import random
import mlflow


################################################################################


# ETL DA BASE

################################################################################



##CONFIGURANDO NUMEROS COM 2 CASAS DECIMAIS
pd.set_option('Float_format','{:.2f}'.format)

## COMPILANDO ARQUIVOS DA PASTA
pasta = 'C:/Users/Guilh/OneDrive/Área de Trabalho/1-MATERIAIS DE ESTUDO/01 - DIVERSOS/01-SCIKIT_LEARN/01-INPUT'
arquivos = glob.glob(os.path.join(pasta, '*.csv'))
base_original = pd.concat([pd.read_csv(i, sep=';', encoding='latin1') for i in arquivos], ignore_index = True)

## REMOVENDO PONTO DA COLUNA
base_original['TU'] = base_original['TU'].str.replace('.','',regex=False)
base_original['TKU'] = base_original['TKU'].str.replace('.','',regex=False)

## ALTERANDO FORMATAÇÃO DA COLUNA
base_original['TKU'] = base_original['TKU'].fillna(0)
base_original['TKU'] = base_original['TKU'].astype(float)
base_original['TU'] = base_original['TU'].astype(int)
base_original['TKU'] = base_original['TKU'].astype(int)
base_original['Mes_Ano'] = pd.to_datetime(base_original['Mes_Ano'], format='%m/%Y')

## CRIANDO COLUNAS
base_original['MÊS'] = base_original['Mes_Ano'].dt.month
base_original['ANO'] = base_original['Mes_Ano'].dt.year

#REMOVENDO DUPLICADOS E NULOS
base_original = base_original.drop_duplicates()

# REMOVENDO COLUNA TKU POIS O OBJETIVO É APNAS TU
base_original = base_original.drop('TKU', axis=1)
base_original = base_original.drop('Mes_Ano', axis=1)

# REMOVENDO VALORES ABAIXO DE 1 TONELADA
base_original = base_original[base_original["TU"]>= 1]

# VENDO A BASE
base_original.head(-5)

,Ferrovia,Mercadoria_ANTT,Estacao_Origem,UF_Origem,Estacao_Destino,UF_Destino,TU,MÊS,ANO
0,EFC,Álcool,Itaqui Base Combustível,MA,Marabá,PA,184,1,2006
1,EFC,Bebidas e Vasilhames,Ponta da Madeira Pêra do Píer,MA,Imperatriz,MA,1636,1,2006
2,EFC,Cobre,Paraupebas,PA,Ponta da Madeira Cobre,MA,24461,1,2006
3,EFC,Ferro Gusa,Açailândia,MA,Ponta da Madeira Pêra do Píer,MA,116272,1,2006
4,EFC,Ferro Gusa,Marabá,PA,Ponta da Madeira Pêra do Píer,MA,205242,1,2006
...,...,...,...,...,...,...,...,...,...
148653,RMS,Soja,Marialva,PR,D Pedro II,PR,179173,4,2023
148654,RMS,Soja,Marialva,PR,São Francisco do Sul,SC,80527,4,2023
148655,RMS,Soja,Maringa,PR,D Pedro II,PR,193917,4,2023
148656,RMS,Soja,Maringa,PR,Rio Grande,RS,1710,4,2023


In [2]:


####################################################################################

#     CRIANDO DATASET DO MODELO

####################################################################################

'''

AQUI VAMOS CRIAR O DATASET DO MODELO
FAZENDO ETL COM AS COLUNAS CATEGORICAS E APLICANDO CÓDIGOS PARA CADA UMA, 
POSTERIORMENTE USANDO AS COLUNAS NUMERICAS REPRESENTADAS COMO VARIAVEIS DO MODELO.
EM VEZ DE USAR O NOME DA FERROVIA ESTAMOS CRIANDO UM CÓDIGO DISTINTO DELA.

'''


# REPRODUTIBILIDADE
np.random.seed(2)


## BASE MODELO
#print(base_original.columns)
print('--------------------')
base_modelo = base_original

# TRANSFORMANDO COLUNAS
base_modelo['ANO'] = base_modelo['ANO'].astype(int)
base_modelo['MÊS'] = base_modelo['MÊS'].astype(int)
base_modelo['TU'] = base_modelo['TU'].astype(int)

#
############### CRIANDO OS CÓDIGOS DE CADA CATEGORIA
# SELECIONA APENAS AS COLUNAS STRING/CATEGORIAS/OBJETO
categ = base_modelo.select_dtypes(include=['object', 'category']).copy()
# LAMBDA QUE TRANSFORMA AS COLUNAS E NUMEROS
cod_categorica_hoje_1 = categ.apply(lambda col: col.astype('category').cat.codes)
# RENOMEANDO AS COLUNAS DE CODIGO COM PREFIXO
cod_categorica_1 = cod_categorica_hoje_1.add_prefix('Codigo_')


# CONCATENANDO AS BASES DE CODIGOS COM A ORIGINAL
codigos_categorias = pd.concat([base_modelo, cod_categorica_1], axis=1)

print('################# BASE COM DE:PARA ########################\n')
# BASE COM TUDO
codigos_categorias.head(1)


##########  PARA VISUALIZAR DE:PARA DE COLUNA ESPECIFICA
#visualizar_de_para = codigos_categorias[['Ferrovia', 'Codigo_Ferrovia']].drop_duplicates().reset_index(drop=True)
#print('\n################# VERIFICAR ########################\n')
#print(visualizar_de_para.head(3))



--------------------
################# BASE COM DE:PARA ########################



,Ferrovia,Mercadoria_ANTT,Estacao_Origem,UF_Origem,Estacao_Destino,UF_Destino,TU,MÊS,ANO,Codigo_Ferrovia,Codigo_Mercadoria_ANTT,Codigo_Estacao_Origem,Codigo_UF_Origem,Codigo_Estacao_Destino,Codigo_UF_Destino
0,EFC,Álcool,Itaqui Base Combustível,MA,Marabá,PA,184,1,2006,0,98,216,5,234,9


In [3]:
####################################################################################

#     TRANSFORMANDO DATASET DO MODELO RANDOMFOREST

####################################################################################

''' 
AQUI ESTOU REMOVENDO OS VALORES MENORES DE 1000 MIL TONELADAS DE MOVIMENTAÇÃO
POIS ABAIXO DE MIL DEIXA O MODELO RUIM, MUITO COMPLEXO PRA ENTENDER OS DADOS
QUE VARIAM DE ZERO A MILHOES.

'''
# CRIANDO A BASE
base_treino = codigos_categorias.select_dtypes(exclude=['object', 'category']).copy()

#REMOVENDO DUPLICADOS
base_treino = base_treino.drop_duplicates()
base_treino = base_treino.fillna(0)

# SALVANDO BASE DO MODELO COMPLETA SEM FILTROS
base_modelo_completa = base_treino

# LIMITANDO A BASE REMOVENDO 2023
base_treino = base_treino[base_treino['ANO'] != 2023][list(base_treino.columns)]


# REMOVENDO OS OUTLIERS - TUDO QUE É MENOR DE MIL TONELADAS
base_treino = base_treino[base_treino['TU'] >= 1000.00]
base_modelo = base_treino


# REMOVENDO OS OUTLIERS - TUDO QUE ESTÁ MAIOR DO 3º QUARTIL EM TONELADAS
#q3 = base_treino['TU'].quantile(0.75)
#base_treino = base_treino[base_treino['TU'] <= q3]
#base_modelo = base_treino



# LOGARITMO BASE10
base_modelo['TU_LOG10'] = np.log10(base_modelo['TU'])


# REMOVENDO COLUNA REAL
base_modelo = base_modelo.drop('TU', axis=1)


base_modelo = base_modelo[["MÊS", "ANO", "Codigo_Ferrovia", "Codigo_Mercadoria_ANTT",
                          "Codigo_Estacao_Origem", "Codigo_UF_Origem", "Codigo_Estacao_Destino",
                          "Codigo_UF_Destino", "TU_LOG10"]]


print('################# BASE MODELO ########################')
print(base_modelo.head(3))
base_modelo.describe()


################# BASE MODELO ########################
   MÊS   ANO  Codigo_Ferrovia  Codigo_Mercadoria_ANTT  Codigo_Estacao_Origem  \
1    1  2006                0                      11                    316   
2    1  2006                0                      22                    293   
3    1  2006                0                      44                     36   

   Codigo_UF_Origem  Codigo_Estacao_Destino  Codigo_UF_Destino  TU_LOG10  
1                 5                     187                  5      3.21  
2                 9                     296                  5      4.39  
3                 5                     297                  5      5.07  


,MÊS,ANO,Codigo_Ferrovia,Codigo_Mercadoria_ANTT,Codigo_Estacao_Origem,Codigo_UF_Origem,Codigo_Estacao_Destino,Codigo_UF_Destino,TU_LOG10
count,109216.00,109216.00,109216.00,109216.00,109216.00,109216.00,109216.00,109216.00,109216.00
mean,6.55,2013.46,6.90,53.71,234.52,10.26,224.13,11.75,4.03
std,3.43,5.00,3.92,30.10,127.57,5.37,121.22,5.59,0.68
min,1.00,2006.00,0.00,0.00,1.00,0.00,0.00,0.00,3.00
25%,4.00,2009.00,3.00,23.00,132.00,6.00,127.00,6.00,3.51
50%,7.00,2013.00,7.00,59.00,247.00,9.00,233.00,13.00,3.92
75%,10.00,2018.00,12.00,81.00,341.00,15.00,340.00,18.00,4.42
max,12.00,2022.00,12.00,101.00,437.00,19.00,410.00,19.00,7.14


In [17]:
''' 
K-MEANS

ESCOLHENDO AS FEATURES/COLUNAS - 
Ao usar este modelo, no momento de escolher as features que vão compor o cluster não devemos escolher colunas que já possuem categorias pré-definidas como exemplo: Codigo_Ferrovia,
Codigo_Mercadoria_ANTT, Codigo_Estacao_Origem, Codigo_UF_Origem, Codigo_Estacao_Destino, Codigo_UF_Destino .... a intenção é usar colunas númericas mas que não estejam pré-classificadas. 
As toneladas por exemplo são um valor númerico sem classificação implicita ou explicita, outras informações que podem ir para o cluster são algumas estatísticas 
agregadas do dataset: média, desvio, tendência, sazonalidade.

Evitar colocar MÊS e ANO direto no K-Means pois faz o cluster separar por calendário, não por comportamento de vendas. O modelo passa a agrupar por tempo, e não por perfil de produto.
O cluster muda todo mês sem refletir mudança real no negócio.

'''


from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score



####### DEFINIDO A UNIDADE QUE GERARÁ O CLUSTER
''' EM NOSSO CASE VAMOS ESCOLHER A COLUNA Codigo_Mercadoria_ANTT POIS QUERO AGRUPAR POR PRODUTOS E SUAS TONELADAS '''



####### AGREGAR OS DADOS POR PRODUTO
base_modelo_agregado = (
    base_modelo.groupby("Codigo_Mercadoria_ANTT").agg(
        tu_media=("TU_LOG10", "mean"),
        tu_std=("TU_LOG10", "std"),
        tu_min=("TU_LOG10", "min"),
        tu_max=("TU_LOG10", "max"),
        n_registros=("TU_LOG10", "count")).fillna(0).reset_index()
        )



####### ESCALONANDO OS DADOS PARA NORMALIZAR/PADRONIZAR A ORDEM DE GRANDEZA
features_cluster = [
    "tu_media",
    "tu_std",
    "tu_min",
    "tu_max",
    "n_registros"
]

x = base_modelo_agregado[features_cluster]
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)



#######  NÃO COLOCAR O NUMERO DE CLUSTER ARBITRARIAMENTE K=5 e GERAR MODELO K-MEANS
scores = []
grupos_possiveis = range(2, 15)

for k in grupos_possiveis:
    kmeans = KMeans(n_clusters=k, random_state=2, n_init=10)
    labels = kmeans.fit_predict(x_scaled)
    scores.append(silhouette_score(x_scaled, labels))
melhor_numero = grupos_possiveis[scores.index(max(scores))]

print('O melhor número de clusters sugerido pelo silhouette_score é: ', melhor_numero)

''' 

USANDO HEURISTICA MATEMATICA - 
aqui usamos a quantidade de amostras (linhas) do dataset escalonado (x_scaled), ou seja, o número total de registros usados no modelo.
Calcula um valor heurístico para o número de clusters (k) usando a raiz quadrada do tamanho do dataset, dividido por 2 e converte para inteiro para ser usado no K-Means.
Apenas a raiz quadrada de N costuma gerar clusters demais, clusters muito pequenos e dividir por 2 ajuda a reduzir a granularidade. Evita “micro-clusters”.
Caso contrário se fosse linear, 10.000 amostras gerariam 10.000 clusters (absurdo) (A raiz controla o crescimento e evita overfragmentação)

#######  NÃO COLOCAR O NUMERO DE CLUSTER ARBITRARIAMENTE K=5
import numpy as np

n_samples = x_scaled.shape[0]
k = int(np.sqrt(n_samples / 2))

'''


####### GERANDO MODELO FINAL COM O NUMERO DE CLUSTER ENCONTRADO NO SILHOUETTE_SCORE
kmeans_com_silhouette = KMeans(n_clusters=melhor_numero, random_state=2, n_init=10)
base_modelo_agregado["cluster_produto"] = kmeans_com_silhouette.fit_predict(x_scaled)


####### INTERPRETAR O CLUSTER DEVOLVIDO PELO MODELO
base_modelo_agregado.groupby("cluster_produto")[features_cluster].mean()
cluster_gerado = base_modelo_agregado['cluster_produto'].unique()
print('clusters gerados pelo k-means:', cluster_gerado)
print('\n############ BASE COM A COLUNA ####################')
base_modelo_agregado.head()




O melhor número de clusters sugerido pelo silhouette_score é:  4
clusters gerados pelo k-means: [0 1 3 2]

############ BASE COM A COLUNA ####################


,Codigo_Mercadoria_ANTT,tu_media,tu_std,tu_min,tu_max,n_registros,cluster_produto
0,0,3.52,0.28,3.01,4.27,66,0
1,1,3.66,0.41,3.00,5.32,1797,1
2,2,3.58,0.37,3.00,4.73,634,0
3,4,3.62,0.37,3.00,4.43,221,0
4,5,3.43,0.30,3.00,4.15,124,0


In [18]:
####### LEVANDO CLUSTER GERADO PELO MODELO AO DATASET ORIGINAL
df_novo = base_modelo.merge(
    base_modelo_agregado[["Codigo_Mercadoria_ANTT", "cluster_produto"]],
    on="Codigo_Mercadoria_ANTT", how="left")

df_novo.head()

,MÊS,ANO,Codigo_Ferrovia,Codigo_Mercadoria_ANTT,Codigo_Estacao_Origem,Codigo_UF_Origem,Codigo_Estacao_Destino,Codigo_UF_Destino,TU_LOG10,cluster_produto
0,1,2006,0,11,316,5,187,5,3.21,0
1,1,2006,0,22,293,9,296,5,4.39,1
2,1,2006,0,44,36,5,297,5,5.07,1
3,1,2006,0,44,252,9,297,5,5.31,1
4,1,2006,0,44,365,5,297,5,3.90,1


In [ ]:
''' 
O K-Means está correto e funcional, mas ainda faltam validações comuns em projetos mais maduros.
Validações que não aparecem ou não estão completas:

Elbow Method (Inertia)
Análise de tamanho e balanceamento dos clusters
Estabilidade dos clusters (seed / amostragem)
Validação temporal (robustez ao longo do tempo)
Análise de outliers / ruído
Interpretabilidade de clusters (perfil médio por cluster)
Visualização dos clusters (ex: PCA / UMAP)
Validação de aderência ao objetivo de negócio

Se fosse só prova técnica, está ok.
Se fosse produção / apresentação sênior, essas validações ainda entram.

'''